In [19]:
import sys
import os
sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]

# Now we can safely import atlasopenmagic
import atlasopenmagic as atom

In [20]:
import uproot # for reading .root files
import time # to measure time to analyse
import math # for mathematical functions such as square root
import awkward as ak # for handling complex and nested data structures efficiently
import numpy as np # # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import MaxNLocator,AutoMinorLocator # for minor ticks
from lmfit.models import PolynomialModel, GaussianModel # for the signal and background fits
import vector #to use vectors
import requests # for HTTP access
import aiohttp # HTTP client support
import pandas as pd

In [21]:
atom.set_release('2025e-13tev-beta')

Release '2025e-13tev-beta' already active with cached metadata.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [22]:
lumi = 36

In [23]:
def get_xsec_weight(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
        / metadata["sumOfWeights"]
    )

def get_N_inclusive(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def get_inclusive_yield(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def calc_weight(xsec_weight, weight_arr, data):
    for variable in weight_arr:
        xsec_weight = xsec_weight * data[variable]
    return xsec_weight

In [24]:
variable_weights_arr = ["mcWeight"]
variables = [] + variable_weights_arr

In [28]:
# Loop over all the files in our list
def check_weights(dsid_list, fraction = 1):
    for dsid in dsid_list:
        sum_of_mc_weights = 0
        N_gen_rel = 0
        N_inclusive = 0
        numevents = 0

        metadata = atom.get_metadata(dsid)
        print(metadata["generator"])
        xsec_weight = get_xsec_weight(metadata, lumi)
        N_inclusive = get_N_inclusive(metadata, lumi)

        file_list = atom.get_urls(dsid, skim = "noskim", protocol='root', cache=False)
        #print("filelist: ", file_list)

        #for url in atom.get_urls(dsid, protocol='root', cache=True):
        #    print("url: ", url)
        for afile in file_list:
                # Print which sample is being processed
            #print(f'Processing file {afile} ({file_list.index(afile)+1}/{len(file_list)})')
            #print("afile: ", f'{afile}')

            # Open file
            tree = uproot.open(afile + ":analysis")

            numevents += tree.num_entries

            # Perform the cuts for each data entry in the tree and calculate the invariant mass
            for data in tree.iterate(variables, library="ak", entry_stop=numevents*fraction):
                #print("data: ", data)

                sum_of_mc_weights = sum_of_mc_weights + ak.sum(data["mcWeight"])
                #print("weights: ", data["mcWeight"])

        N_gen_rel = xsec_weight * sum_of_mc_weights
        print('Done processing data of dsid ', dsid)
        #print("sum_of_weights: ", metadata["sumOfWeights"])
        #print("sum_of_mc_weights (calculated sum of mc_weights): ", sum_of_mc_weights)
        #print("N_gen_rel (calculated sum of xsec_weights): ", N_gen_rel)
        print("N_inclusive: ", N_inclusive)
        print("Number of events in sample: ", numevents)
    return sum_of_mc_weights

In [29]:
dsid_list = [301209, 700323, 700324, 700325, 700470, 700471, 700472, 410219, 700589, 700602, 601624, 601628]

In [30]:
check_weights(dsid_list)

Pythia8(v8.186)+EvtGen(v1.2.0)
Done processing data of dsid  301209
N_inclusive:  63.784800000000004
Number of events in sample:  19928
Sherpa
Done processing data of dsid  700323
N_inclusive:  1950741.306252
Number of events in sample:  3994786
Sherpa
Done processing data of dsid  700324
N_inclusive:  10398083.004
Number of events in sample:  20869901
Sherpa
Done processing data of dsid  700325
N_inclusive:  67683331.75572
Number of events in sample:  85036532
Sherpa
Done processing data of dsid  700470
N_inclusive:  534325.44816
Number of events in sample:  664668
Sherpa
Done processing data of dsid  700471
N_inclusive:  11304959.67864
Number of events in sample:  3664380
Sherpa
Done processing data of dsid  700472
N_inclusive:  77680565.07552001
Number of events in sample:  38413710
MadGraph5_aMC@NLO(v2.3.3.p0)+Pythia8(v8.210)+EvtGen(v1.2.0)
Done processing data of dsid  410219
N_inclusive:  1486.3968000000002
Number of events in sample:  119613
Sherpa(v.2.2.12.f290b9)
Done processi

np.float32(1.478547e+06)